Baseline训练

In [3]:
from ultralytics import YOLO
import os

# ================= 配置区域 =================
# 1. 确保这里指向刚才生成的 NEU-DET 配置文件
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 2. 检查一下文件是否存在，防止报错
if not os.path.exists(NEU_YAML):
    print(f"❌ 错误: 找不到配置文件 {NEU_YAML}")
else:
    print(f"✅ 配置文件就绪: {NEU_YAML}")

    # ================= 开始训练 =================
    print("🚀 开始 Exp 1: Baseline 训练...")
    
    # 加载官方的 YOLOv11 nano 模型 (ImageNet 预训练)
    # 如果下载慢，它会自动从 GitHub 拉取
    model = YOLO('yolo11n.pt') 

    # 训练参数
    model.train(
        data=NEU_YAML,
        epochs=50,             # 训练 50 轮
        batch=16,              # 显存如果不够(比如报错CUDA OOM)，改成 8
        imgsz=640,             # 标准输入尺寸
        workers=4,             # 数据加载线程数
        project='result_exp1',  # 实验结果保存在这个文件夹
        name='1_Baseline',     # 本次实验的子文件夹名
        device='0',            # 使用 0 号 GPU
        exist_ok=True          # 如果文件夹已存在，允许覆盖
    )
    
    print("\n🏆 Baseline 训练完成！")
    print(f"查看结果请前往: Thesis_Exp/1_Baseline")

✅ 配置文件就绪: /root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml
🚀 开始 Exp 1: Baseline 训练...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=1_Baseline, nbs=64, nms=

迁移学习的源域训练（可惜效果不是很完美）

In [10]:
from ultralytics import YOLO
import os

# ================= 配置区域 =================
# 指向 Severstal 配置文件
SEV_YAML = "/root/autodl-tmp/exp1/SEVERSTAL_YOLO/severstal.yaml"

# ================= 开始训练 =================
print("🚀 开始 Exp 2: Severstal 域适应预训练...")

# 依然从 ImageNet 权重开始
model = YOLO('yolo11n.pt') 

# 训练参数
model.train(
    data=SEV_YAML,
    epochs=30,             # 数据量大，30轮足够模型学到特征
    batch=64,              # 4090 显存充足，开大 Batch 加速
    imgsz=640,
    workers=8,             # 增加数据加载线程
    project='result_exp1',  
    name='2_Severstal_Pretrain', # 实验名称
    device='0',
    exist_ok=True,
    val=False              # ⚠️ 关键点：预训练阶段我们只在乎特征提取，不强求验证集分数，关闭验证可提速 30%
)

print("\n🏆 Severstal 预训练完成！")
# 这里的 best.pt 是我们下一步 Exp 3 和 Exp 4 的核心资产
print(f"核心权重已保存:/2_Severstal_Pretrain/weights/best.pt")

🚀 开始 Exp 2: Severstal 域适应预训练...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/SEVERSTAL_YOLO/severstal.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=2_Severstal_Pretrain, nbs=64, nms=False, opset=None, optimize=False, opti

In [5]:
import os
cache_path = "/root/autodl-tmp/exp1/SEVERSTAL_YOLO/labels/train.cache"
if os.path.exists(cache_path):
    os.remove(cache_path)
    print("🗑️ 已删除脏缓存，下次训练会重新扫描。")
else:
    print("✅ 缓存不存在，无需删除。")

🗑️ 已删除脏缓存，下次训练会重新扫描。


迁移学习（负迁移）--->暂时放弃迁移，先进行隐私保护

In [11]:
from ultralytics import YOLO
import os

# ================= 配置区域 =================
# 1. 指向 NEU-DET (我们要回目标数据集了)
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 2. 关键！加载 Exp 2 训练好的权重
# 请确保这个路径是存在的
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt"

# ================= 开始训练 =================
print(f"🚀 开始 Exp 3: Clean Transfer (迁移学习)...")
print(f"Loading weights from: {PRETRAINED_WEIGHTS}")

# 加载预训练权重
model = YOLO(PRETRAINED_WEIGHTS) 

# 训练参数
model.train(
    data=NEU_YAML,
    epochs=50,             # 50 轮微调
    batch=16,              
    imgsz=640,
    workers=4,
    project='Thesis_Exp',
    name='3_Clean_Transfer', # 实验名
    device='0',
    exist_ok=True,
    val=True               # ✅ 这里要把验证打开，我们要看最终分数！
)

print("\n🏆 Exp 3 迁移学习完成！")
print("请对比 Exp 1 (Baseline) 和 Exp 3 (Transfer) 的 mAP50 分数。")

🚀 开始 Exp 3: Clean Transfer (迁移学习)...
Loading weights from: /root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/2_Severst

冻结一些参数再次进行迁移学习，然而效果仍然一般

In [12]:
from ultralytics import YOLO

# ================= 配置 =================
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
# 依然使用 Exp 2 的权重
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt"

print(f"🚀 开始 Exp 3 (Fix): 冻结骨干 + 低学习率微调...")

model = YOLO(PRETRAINED_WEIGHTS) 

model.train(
    data=NEU_YAML,
    epochs=50,
    batch=16,
    imgsz=640,
    workers=4,
    project='result_exp1',
    name='3_Transfer_Fixed', # 改个名，区分一下
    device='0',
    exist_ok=True,
    val=True,
    
    # 🔥 核心改进 1: 冻结骨干层
    # YOLOv11n 的骨干网大约在前 10 层。
    # 我们锁住它，保护 Severstal 学到的特征不被破坏。
    freeze=10, 
    
    # 🔥 核心改进 2: 降低学习率
    # 相比默认的 0.01，我们将初始学习率降低 10 倍
    # 这样模型会更“温柔”地适应新数据
    lr0=0.001,
    lrf=0.01   # 最终学习率也相应降低
)

print("\n🏆 Exp 3 (修正版) 完成！")

🚀 开始 Exp 3 (Fix): 冻结骨干 + 低学习率微调...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/2_Severstal_Pretrain/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=3_Transfer_Fixe

在BaseLine的基础上第一轮隐私保护训练，效果很差，原因是：“噪声淹没 (Noise Overwhelming)”

In [14]:
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ================= 1. 修正版 SA-LDP 隐私引擎 =================
class SA_LDP_Engine:
    def __init__(self, model, epsilon=10.0, delta=1e-5, beta=0.5):
        self.model = model
        self.epsilon = epsilon
        self.delta = delta
        self.beta = beta
        self.layer_roles = self._identify_layers()
        # ❌ 删除 self.device = ... 避免初始化时的误判

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self):
        # 1. 动态捕获当前参数所在的设备 (这是修复的关键！)
        # 我们随便找一个有梯度的参数来确定当前设备
        current_device = None
        for p in self.model.parameters():
            if p.requires_grad:
                current_device = p.device
                break
        
        if current_device is None: return # 如果没有参数需要更新，直接跳过

        sensitivities = []
        param_groups = []
        names_list = []

        # 2. 收集梯度
        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sensitivities.append(p.grad.norm(2).item())
                param_groups.append(p)
                names_list.append(name)
        
        if not sensitivities: return

        # 3. 预算分配 (确保 Tensor 在正确的 device 上)
        sens_tensor = torch.tensor(sensitivities, device=current_device)
        
        factors = []
        for n in names_list:
            role = self.layer_roles.get(n, "neck")
            if role == "backbone": factors.append(0.5) 
            elif role == "head":   factors.append(2.0) 
            else:                  factors.append(1.0)
            
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        # 4. 注入噪声
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            
            # 梯度裁剪
            clip_val = 1.0
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            # 计算噪声标准差
            c = np.sqrt(2 * np.log(1.25 / self.delta))
            sigma = c * clip_val / (layer_eps + 1e-8)
            
            # ✅ 修复: 使用 randn_like 确保噪声和参数在同一个设备，且形状一致
            # noise = N(0,1) * sigma
            noise = torch.randn_like(p.grad) * sigma
            
            p.grad.add_(noise)
            idx += 1

# ================= 2. 训练器保持不变 (但需要重新运行定义) =================
class PrivacyTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        # 提高 epsilon 到 15.0 试试，让模型更容易收敛一点
        self.privacy_engine = SA_LDP_Engine(model, epsilon=15.0, delta=1e-5) 
        print(f"🛡️ SA-LDP 隐私引擎已挂载! Epsilon={self.privacy_engine.epsilon}")
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step() 
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()

# ================= 3. 重新运行 Exp 4 =================
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
weights_path = 'yolo11n.pt' 

print("\n🚀 (Retry) 开始 Exp 4: SA-LDP 隐私保护训练...")

trainer = PrivacyTrainer(overrides={
    'model': weights_path,
    'data': NEU_YAML,
    'epochs': 50,
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '4_Ours_SA_LDP',
    'device': '0',
    'val': True,
    'workers': 4
})

trainer.train()


🚀 (Retry) 开始 Exp 4: SA-LDP 隐私保护训练...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=4_Ours_SA_LDP2, nbs=64, nms=False, opset=None, optimize=False, optimiz

KeyboardInterrupt: 

改进：

加入 Warm-up (热身机制)：前 5 轮完全不加噪声。让模型先找到一个不错的“坑”，然后再开始加噪声。这能避免一开始就被噪声带偏到沟里去。

大幅放宽 Epsilon：从 15 提到了 50。先保证能训练，再去谈隐私。能跑通了我们再慢慢往回调。

放宽梯度裁剪 (Clip)：从 1.0 提到 5.0。YOLO 的梯度本身很大，切得太狠（1.0）会把有效信息全切没了

更新EMA 模型 (Exponential Moving Average)：负责验证和测试，它是主模型的平滑版本，性能更稳定。

作为BaseLine的隐私保护，已经达成了目标！！

In [16]:
import torch
import torch.nn as nn
import numpy as np
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer

# ================= 1. SA-LDP 引擎 (逻辑保持 V3 不变) =================
class SA_LDP_Engine:
    def __init__(self, model, epsilon=50.0, delta=1e-5, beta=0.5):
        self.model = model
        self.epsilon = epsilon
        self.delta = delta
        self.beta = beta
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        # Warm-up: 前 3 轮不加噪声 (从5轮缩减到3轮，快速验证效果)
        if current_epoch < 3:
            return

        # 动态获取设备
        current_device = next(self.model.parameters()).device
        
        sensitivities = []
        param_groups = []
        names_list = []

        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sensitivities.append(p.grad.norm(2).item())
                param_groups.append(p)
                names_list.append(name)
        
        if not sensitivities: return

        # 预算分配
        factors = []
        for n in names_list:
            role = self.layer_roles.get(n, "neck")
            if role == "backbone": factors.append(0.5) 
            elif role == "head":   factors.append(2.0) 
            else:                  factors.append(1.0)
            
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            
            # 梯度裁剪
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            
            c = np.sqrt(2 * np.log(1.25 / self.delta))
            sigma = c * clip_val / (layer_eps + 1e-8)
            
            # 注入噪声
            noise = torch.randn_like(p.grad) * sigma
            p.grad.add_(noise)
            idx += 1

# ================= 2. 训练器 (修复 EMA 更新 bug) =================
class PrivacyTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = SA_LDP_Engine(model, epsilon=50.0, delta=1e-5) 
        print(f"🛡️ SA-LDP V4.0 (EMA Fixed) 已挂载!")
        return model

    def optimizer_step(self):
        """完全模拟原版 step 逻辑，仅插入隐私操作"""
        self.scaler.unscale_(self.optimizer)
        
        # 1. 注入隐私噪声 (仅修改梯度)
        self.privacy_engine.step(self.epoch) 
        
        # 2. 原版也会在这里做一次全局裁剪，我们可以保留作为双重保险
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=10.0)
        
        # 3. 参数更新
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        
        # 🔥🔥🔥 关键修复：更新 EMA 模型！🔥🔥🔥
        # 如果没有这一行，验证集永远测的是初始化的烂模型
        if self.ema:
            self.ema.update(self.model)

# ================= 3. 执行 Exp 4 =================
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"
weights_path = 'yolo11n.pt' 

print("\n🚀 (V4.0) 开始 Exp 4: 修复 EMA 更新问题...")

trainer = PrivacyTrainer(overrides={
    'model': weights_path,
    'data': NEU_YAML,
    'epochs': 30,         # 先跑30轮看趋势
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1',
    'name': '4_Ours_Fixed_EMA',
    'device': '0',
    'val': True,
    'workers': 4,
    'warmup_epochs': 3.0  # 显式配合 warmup
})

trainer.train()


🚀 (V4.0) 开始 Exp 4: 修复 EMA 更新问题...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=4_Ours_Fixed_EMA, nbs=64, nms=False, opset=None, optimize=False, optimize

In [8]:
# 在 Python 终端执行
import numpy
import cv2
import ultralytics
print(f"NumPy 版本: {numpy.__version__}")
print(f"OpenCV 版本: {cv2.__version__}")
print("依赖导入正常 ✅")

NumPy 版本: 1.26.4
OpenCV 版本: 4.8.1
依赖导入正常 ✅


In [1]:
import ultralytics
import torch
import numpy as np
import cv2

# 验证版本
print(f"Ultralytics 版本：{ultralytics.__version__}")  # 需≥8.2.89
print(f"PyTorch 版本：{torch.__version__}")
print(f"CUDA 可用：{torch.cuda.is_available()}")
print(f"NumPy 版本：{np.__version__}")
print(f"OpenCV 版本：{cv2.__version__}")

# 验证 YOLO11 加载
model = ultralytics.YOLO('yolo11n.pt')

Ultralytics 版本：8.3.248
PyTorch 版本：2.1.2+cu118
CUDA 可用：True
NumPy 版本：1.26.4
OpenCV 版本：4.8.1


以下是攻击验证，加入噪声前后，攻击最脆弱的第一层卷积层 (Conv1)
由于深度神经网络的逐层信息处理特性，第一层卷积层（Input Layer）直接接触原始像素，包含最多的纹理和空间信息。如果第一层都无法被防御，深层防御则无从谈起。因此，我们在实验中重点攻击模型的第一层卷积梯度，以验证防御的最坏情况（Worst-case Evaluation）。

In [18]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO
import glob
import os

# ================= 配置 =================
# 自动寻找一张存在的图片
SEARCH_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO/images/train"
found_imgs = glob.glob(os.path.join(SEARCH_DIR, "*.jpg"))

if len(found_imgs) > 0:
    IMG_PATH = found_imgs[0] # 取第一张
    print(f"🎯 锁定攻击目标: {IMG_PATH}")
else:
    print(f"❌ 错误: 在 {SEARCH_DIR} 找不到图片，请检查路径！")
    # 这里的 IMG_PATH 会导致后续报错，所以如果没找到图，请手动修改路径
    IMG_PATH = "error_no_image_found"

# ================= 工具函数 =================
def preprocess_image(img_path, device):
    """读取图片并缩放"""
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((64, 64)), # 攻击 64x64 的缩略图，速度快且效果明显
        transforms.ToTensor(),
    ])
    return transform(img).unsqueeze(0).to(device)

def reconstruct(model, target_grad, device, defense_noise=0.0):
    """核心攻击逻辑: DLG (Deep Leakage from Gradients)"""
    # 初始化假图片 (Dummy Data)
    dummy_data = torch.randn(1, 3, 64, 64, device=device).requires_grad_(True)
    
    # LBFGS 优化器
    optimizer = torch.optim.LBFGS([dummy_data])
    
    print(f"⚡ 开始攻击 (防御强度 Noise={defense_noise})...")
    
    # 迭代攻击 100 次
    for i in range(100): 
        def closure():
            optimizer.zero_grad()
            
            # 假图片前向传播
            dummy_pred = model(dummy_data) 
            dummy_loss = dummy_pred.mean()
            
            # 计算假梯度
            # create_graph=True 是为了允许对梯度进行微分 (高阶导数)
            dummy_grad = torch.autograd.grad(dummy_loss, model.parameters(), create_graph=True)
            
            # 计算梯度距离 (MSE)
            grad_diff = 0
            for dg, tg in zip(dummy_grad, target_grad):
                grad_diff += ((dg - tg) ** 2).sum()
            
            grad_diff.backward()
            return grad_diff
        
        optimizer.step(closure)
        
        if i % 20 == 0:
            current_loss = closure().item()
            print(f"   Iter {i}: Gradient Distance {current_loss:.6f}")
            
    return dummy_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()

# ================= 主程序 =================
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("🚀 正在加载模型...")
# 加载 YOLO
full_model = YOLO('yolo11n.pt').model.to(device)

# 🔥🔥🔥 关键修复：强制开启梯度计算！🔥🔥🔥
for p in full_model.parameters():
    p.requires_grad_(True)

# 提取第一层卷积 (最容易泄露隐私的地方)
# YOLOv11/v8 的结构中，model[0] 通常是 Conv 层
target_layer = full_model.model[0] 
# 确保这一层处于评估模式 (关掉 Dropout/BatchNorm 的随机性)
target_layer.eval() 

# 1. 准备真数据
try:
    gt_data = preprocess_image(IMG_PATH, device)
except Exception as e:
    print(f"❌ 读取图片失败: {e}")
    exit()

# 2. 获取【真梯度】(Ground Truth Gradient)
# 前向传播
pred = target_layer(gt_data)
# 模拟 Loss (简单用均值代替)
loss = pred.mean()
# 反向传播计算梯度
gt_grads = torch.autograd.grad(loss, target_layer.parameters())

print("✅ 真梯度获取成功！开始攻击...")

# -------------------------------------------------
# ⚔️ 场景 A: 攻击 Baseline (无噪声)
# -------------------------------------------------
print("\n[Scenario 1] Attacking Baseline (No Privacy)...")
recovered_baseline = reconstruct(target_layer, gt_grads, device, defense_noise=0.0)

# -------------------------------------------------
# 🛡️ 场景 B: 攻击 SA-LDP (有噪声)
# -------------------------------------------------
print("\n[Scenario 2] Attacking SA-LDP (With Privacy)...")

# 模拟 SA-LDP: 给梯度加上高斯噪声
# 这里的噪声水平对应我们 Exp 4 中的设置
noised_grads = []
noise_level = 0.2 # 加大一点噪声，让对比更明显
for g in gt_grads:
    noise = torch.randn_like(g) * noise_level
    noised_grads.append(g + noise)

recovered_saldp = reconstruct(target_layer, noised_grads, device, defense_noise=noise_level)

# ================= 画图展示 =================
print("\n🎨 正在绘制攻击结果图...")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始图
gt_img = gt_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()
# 简单的归一化显示
gt_img = (gt_img - gt_img.min()) / (gt_img.max() - gt_img.min())
axes[0].imshow(gt_img)
axes[0].set_title("Original Image\n(Ground Truth)", fontsize=14)
axes[0].axis('off')

# 攻击无防御
rec_base = (recovered_baseline - recovered_baseline.min()) / (recovered_baseline.max() - recovered_baseline.min())
axes[1].imshow(rec_base)
axes[1].set_title("Attack Baseline\n(DANGER: Leaked!)", fontsize=14, color='red')
axes[1].axis('off')

# 攻击有防御
rec_priv = (recovered_saldp - recovered_saldp.min()) / (recovered_saldp.max() - recovered_saldp.min())
axes[2].imshow(rec_priv)
axes[2].set_title("Attack SA-LDP\n(SAFE: Protected)", fontsize=14, color='green')
axes[2].axis('off')

plt.tight_layout()
save_path = "/root/autodl-tmp/exp1/Thesis_Exp/attack_result_final.png"
plt.savefig(save_path)
print(f"✅ 攻击对比图已保存至: {save_path}")
plt.show()

🎯 锁定攻击目标: /root/autodl-tmp/exp1/NEU_DET_YOLO/images/train/rolled-in_scale_75.jpg
🚀 正在加载模型...
✅ 真梯度获取成功！开始攻击...

[Scenario 1] Attacking Baseline (No Privacy)...
⚡ 开始攻击 (防御强度 Noise=0.0)...
   Iter 0: Gradient Distance 16.946064
   Iter 20: Gradient Distance 0.048879
   Iter 40: Gradient Distance 0.004763
   Iter 60: Gradient Distance 0.001102
   Iter 80: Gradient Distance 0.000359

[Scenario 2] Attacking SA-LDP (With Privacy)...
⚡ 开始攻击 (防御强度 Noise=0.2)...
   Iter 0: Gradient Distance 34.601303
   Iter 20: Gradient Distance 5.043173
   Iter 40: Gradient Distance 4.300262
   Iter 60: Gradient Distance 4.050266
   Iter 80: Gradient Distance 3.924174

🎨 正在绘制攻击结果图...
✅ 攻击对比图已保存至: /root/autodl-tmp/exp1/Thesis_Exp/attack_result_final.png


<Figure size 1500x500 with 3 Axes>

In [19]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO
import glob
import os

# ================= 1. 配置区域 =================
# 依然尝试自动寻找图片
SEARCH_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO/images/train"
found_imgs = glob.glob(os.path.join(SEARCH_DIR, "*.jpg"))

if len(found_imgs) > 0:
    # 稍微换一张图试试，有时候特定的纹理更容易还原
    IMG_PATH = found_imgs[10] if len(found_imgs) > 10 else found_imgs[0]
    print(f"🎯 锁定攻击目标: {IMG_PATH}")
else:
    print("❌ 找不到图片，请检查路径")
    exit()

# ================= 2. 增强版工具函数 (加入 TV 正则化) =================
def preprocess_image(img_path, device):
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((64, 64)), # 保持 64x64
        transforms.ToTensor(),
    ])
    return transform(img).unsqueeze(0).to(device)

def total_variation_loss(img, weight):
    """TV 正则化：让图像变平滑，去除雪花噪点"""
    bs_img, c_img, h_img, w_img = img.size()
    tv_h = torch.pow(img[:,:,1:,:] - img[:,:,:-1,:], 2).sum()
    tv_w = torch.pow(img[:,:,:,1:] - img[:,:,:,:-1], 2).sum()
    return weight * (tv_h + tv_w)

def reconstruct(model, target_grad, device, defense_noise=0.0, use_tv=True):
    """
    V3.0 攻击逻辑：加入 TV Loss 和更多的迭代次数
    """
    # 初始化：这次我们用均值灰度初始化，而不是纯随机，帮助收敛
    dummy_data = torch.zeros(1, 3, 64, 64, device=device).requires_grad_(True)
    nn.init.uniform_(dummy_data, 0, 1) # 稍微给点随机扰动
    
    # 这里的 LBFGS 学习率是关键，稍微调大一点
    optimizer = torch.optim.LBFGS([dummy_data], lr=1.0)
    
    print(f"⚡ 开始攻击 (Noise={defense_noise}, TV={use_tv})...")
    
    # 增加迭代次数到 300
    for i in range(300): 
        def closure():
            optimizer.zero_grad()
            
            dummy_pred = model(dummy_data) 
            dummy_loss = dummy_pred.mean()
            
            # 计算假梯度
            dummy_grad = torch.autograd.grad(dummy_loss, model.parameters(), create_graph=True)
            
            # 1. 梯度匹配损失 (Gradient Matching Loss)
            grad_diff = 0
            for dg, tg in zip(dummy_grad, target_grad):
                grad_diff += ((dg - tg) ** 2).sum()
            
            # 2. TV 正则化损失 (这是去噪的关键！)
            if use_tv:
                tv_loss = total_variation_loss(dummy_data, weight=1e-3)
                total_loss = grad_diff + tv_loss
            else:
                total_loss = grad_diff
            
            total_loss.backward()
            return total_loss
        
        optimizer.step(closure)
        
        # 每 50 次打印一下损失
        if i % 50 == 0:
            current_loss = closure().item()
            print(f"   Iter {i}: Loss {current_loss:.6f}")
            
    return dummy_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()

# ================= 3. 主程序 =================
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("🚀 加载模型与数据...")
full_model = YOLO('yolo11n.pt').model.to(device)
for p in full_model.parameters(): p.requires_grad_(True)
target_layer = full_model.model[0] 
target_layer.eval()

# 准备真数据
gt_data = preprocess_image(IMG_PATH, device)
pred = target_layer(gt_data)
loss = pred.mean()
gt_grads = torch.autograd.grad(loss, target_layer.parameters())

# -------------------------------------------------
# ⚔️ 场景 A: 攻击 Baseline (无噪声)
# -------------------------------------------------
print("\n[Scenario 1] Attacking Baseline (Expecting Clear Image)...")
# use_tv=True 是让中间图变清晰的关键
recovered_baseline = reconstruct(target_layer, gt_grads, device, defense_noise=0.0, use_tv=True)

# -------------------------------------------------
# 🛡️ 场景 B: 攻击 SA-LDP (有噪声)
# -------------------------------------------------
print("\n[Scenario 2] Attacking SA-LDP (Expecting Noise)...")

noised_grads = []
# ⚠️ 这里 noise_level 设为 0.1 试试 (如果 0.2 太大导致全是乱码，0.1 更能看出隐约的轮廓被破坏)
noise_level = 0.1 
for g in gt_grads:
    noise = torch.randn_like(g) * noise_level
    noised_grads.append(g + noise)

recovered_saldp = reconstruct(target_layer, noised_grads, device, defense_noise=noise_level, use_tv=True)

# ================= 画图 =================
print("\n🎨 绘制结果...")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始图
gt_img = gt_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()
gt_img = (gt_img - gt_img.min()) / (gt_img.max() - gt_img.min())
axes[0].imshow(gt_img)
axes[0].set_title("Original Image\n(Ground Truth)")
axes[0].axis('off')

# 攻击无防御
rec_base = (recovered_baseline - recovered_baseline.min()) / (recovered_baseline.max() - recovered_baseline.min())
axes[1].imshow(rec_base)
axes[1].set_title("Attack Baseline\n(DANGER: Leaked!)", color='red')
axes[1].axis('off')

# 攻击有防御
rec_priv = (recovered_saldp - recovered_saldp.min()) / (recovered_saldp.max() - recovered_saldp.min())
axes[2].imshow(rec_priv)
axes[2].set_title("Attack SA-LDP\n(SAFE: Protected)", color='green')
axes[2].axis('off')

save_path = "/root/autodl-tmp/exp1/Thesis_Exp/attack_result_v3.png"
plt.savefig(save_path)
print(f"✅ 结果已保存: {save_path}")
plt.show()

🎯 锁定攻击目标: /root/autodl-tmp/exp1/NEU_DET_YOLO/images/train/rolled-in_scale_64.jpg
🚀 加载模型与数据...

[Scenario 1] Attacking Baseline (Expecting Clear Image)...
⚡ 开始攻击 (Noise=0.0, TV=True)...
   Iter 0: Loss 0.918829
   Iter 50: Loss 0.004929
   Iter 100: Loss 0.004824
   Iter 150: Loss 0.004804
   Iter 200: Loss 0.004778
   Iter 250: Loss 0.004754

[Scenario 2] Attacking SA-LDP (Expecting Noise)...
⚡ 开始攻击 (Noise=0.1, TV=True)...
   Iter 0: Loss 5.260660
   Iter 50: Loss 4.274050
   Iter 100: Loss 4.267358
   Iter 150: Loss 4.267356
   Iter 200: Loss 4.267356
   Iter 250: Loss 4.267356

🎨 绘制结果...
✅ 结果已保存: /root/autodl-tmp/exp1/Thesis_Exp/attack_result_v3.png


<Figure size 1500x500 with 3 Axes>

In [24]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from torchvision import transforms
from ultralytics import YOLO
import glob
import os

# ================= 1. 自动寻找 Scratches 图片 =================
SEARCH_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO/images/train"
LABEL_DIR = "/root/autodl-tmp/exp1/NEU_DET_YOLO/labels/train"

def find_scratch_img():
    print("🔍 正在寻找 Scratches (类别ID=5) 的图片...")
    # NEU-DET 类别顺序: crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches
    # 所以 Scratches 是 5
    txts = glob.glob(os.path.join(LABEL_DIR, "*.txt"))
    for txt in txts:
        with open(txt, 'r') as f:
            for line in f:
                if line.startswith("5 "): # 找到以 5 开头的行
                    img_name = os.path.basename(txt).replace(".txt", ".jpg")
                    img_path = os.path.join(SEARCH_DIR, img_name)
                    if os.path.exists(img_path):
                        return img_path
    return None

IMG_PATH = find_scratch_img()
if not IMG_PATH:
    print("⚠️ 没找到 Scratches 图片，随机用第一张...")
    IMG_PATH = glob.glob(os.path.join(SEARCH_DIR, "*.jpg"))[0]
else:
    print(f"🎯 锁定 Scratches 目标: {IMG_PATH}")

# ================= 2. 增强版工具函数 (加入 TV 正则化) =================
def preprocess_image(img_path, device):
    img = Image.open(img_path).convert('RGB')
    transform = transforms.Compose([
        transforms.Resize((64, 64)), # 保持 64x64
        transforms.ToTensor(),
    ])
    return transform(img).unsqueeze(0).to(device)

def total_variation_loss(img, weight):
    """TV 正则化：让图像变平滑，去除雪花噪点"""
    bs_img, c_img, h_img, w_img = img.size()
    tv_h = torch.pow(img[:,:,1:,:] - img[:,:,:-1,:], 2).sum()
    tv_w = torch.pow(img[:,:,:,1:] - img[:,:,:,:-1], 2).sum()
    return weight * (tv_h + tv_w)

def reconstruct(model, target_grad, device, defense_noise=0.0, use_tv=True):
    """
    V3.0 攻击逻辑：加入 TV Loss 和更多的迭代次数
    """
    # 初始化：这次我们用均值灰度初始化，而不是纯随机，帮助收敛
    dummy_data = torch.zeros(1, 3, 64, 64, device=device).requires_grad_(True)
    nn.init.uniform_(dummy_data, 0, 1) # 稍微给点随机扰动
    
    # 这里的 LBFGS 学习率是关键，稍微调大一点
    optimizer = torch.optim.LBFGS([dummy_data], lr=1.0)
    
    print(f"⚡ 开始攻击 (Noise={defense_noise}, TV={use_tv})...")
    
    # 增加迭代次数到 300
    for i in range(300): 
        def closure():
            optimizer.zero_grad()
            
            dummy_pred = model(dummy_data) 
            dummy_loss = dummy_pred.mean()
            
            # 计算假梯度
            dummy_grad = torch.autograd.grad(dummy_loss, model.parameters(), create_graph=True)
            
            # 1. 梯度匹配损失 (Gradient Matching Loss)
            grad_diff = 0
            for dg, tg in zip(dummy_grad, target_grad):
                grad_diff += ((dg - tg) ** 2).sum()
            
            # 2. TV 正则化损失 (这是去噪的关键！)
            if use_tv:
                tv_loss = total_variation_loss(dummy_data, weight=1e-3)
                total_loss = grad_diff + tv_loss
            else:
                total_loss = grad_diff
            
            total_loss.backward()
            return total_loss
        
        optimizer.step(closure)
        
        # 每 50 次打印一下损失
        if i % 50 == 0:
            current_loss = closure().item()
            print(f"   Iter {i}: Loss {current_loss:.6f}")
            
    return dummy_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()

# ================= 3. 主程序 =================
device = "cuda:0" if torch.cuda.is_available() else "cpu"

print("🚀 加载模型与数据...")
full_model = YOLO('yolo11n.pt').model.to(device)
for p in full_model.parameters(): p.requires_grad_(True)
target_layer = full_model.model[0] 
target_layer.eval()

# 准备真数据
gt_data = preprocess_image(IMG_PATH, device)
pred = target_layer(gt_data)
loss = pred.mean()
gt_grads = torch.autograd.grad(loss, target_layer.parameters())

# -------------------------------------------------
# ⚔️ 场景 A: 攻击 Baseline (无噪声)
# -------------------------------------------------
print("\n[Scenario 1] Attacking Baseline (Expecting Clear Image)...")
# use_tv=True 是让中间图变清晰的关键
recovered_baseline = reconstruct(target_layer, gt_grads, device, defense_noise=0.0, use_tv=True)

# -------------------------------------------------
# 🛡️ 场景 B: 攻击 SA-LDP (有噪声)
# -------------------------------------------------
print("\n[Scenario 2] Attacking SA-LDP (Expecting Noise)...")

noised_grads = []
# ⚠️ 这里 noise_level 设为 0.1 试试 (如果 0.2 太大导致全是乱码，0.1 更能看出隐约的轮廓被破坏)
noise_level = 0.1 
for g in gt_grads:
    noise = torch.randn_like(g) * noise_level
    noised_grads.append(g + noise)

recovered_saldp = reconstruct(target_layer, noised_grads, device, defense_noise=noise_level, use_tv=True)

# ================= 画图 =================
print("\n🎨 绘制结果...")
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始图
gt_img = gt_data.detach().cpu().squeeze().permute(1, 2, 0).numpy()
gt_img = (gt_img - gt_img.min()) / (gt_img.max() - gt_img.min())
axes[0].imshow(gt_img)
axes[0].set_title("Original Image\n(Ground Truth)")
axes[0].axis('off')

# 攻击无防御
rec_base = (recovered_baseline - recovered_baseline.min()) / (recovered_baseline.max() - recovered_baseline.min())
axes[1].imshow(rec_base)
axes[1].set_title("Attack Baseline\n(DANGER: Leaked!)", color='red')
axes[1].axis('off')

# 攻击有防御
rec_priv = (recovered_saldp - recovered_saldp.min()) / (recovered_saldp.max() - recovered_saldp.min())
axes[2].imshow(rec_priv)
axes[2].set_title("Attack SA-LDP\n(SAFE: Protected)", color='green')
axes[2].axis('off')

save_path = "/root/autodl-tmp/exp1/Thesis_Exp/attack_result_v3.png"
plt.savefig(save_path)
print(f"✅ 结果已保存: {save_path}")
plt.show()

🔍 正在寻找 Scratches (类别ID=5) 的图片...
🎯 锁定 Scratches 目标: /root/autodl-tmp/exp1/NEU_DET_YOLO/images/train/scratches_121.jpg
🚀 加载模型与数据...

[Scenario 1] Attacking Baseline (Expecting Clear Image)...
⚡ 开始攻击 (Noise=0.0, TV=True)...
   Iter 0: Loss 1.094822
   Iter 50: Loss 0.011125
   Iter 100: Loss 0.010054
   Iter 150: Loss 0.009833
   Iter 200: Loss 0.009810
   Iter 250: Loss 0.009807

[Scenario 2] Attacking SA-LDP (Expecting Noise)...
⚡ 开始攻击 (Noise=0.1, TV=True)...
   Iter 0: Loss 4.858598
   Iter 50: Loss 3.802995
   Iter 100: Loss 3.784353
   Iter 150: Loss 3.784338
   Iter 200: Loss 3.784338
   Iter 250: Loss 3.784338

🎨 绘制结果...
✅ 结果已保存: /root/autodl-tmp/exp1/Thesis_Exp/attack_result_v3.png


<Figure size 1500x500 with 3 Axes>